# ABD on Colab — tracking, feature extraction, training

**Trước khi chạy** (một lần): tạo thư mục `MyDrive/abd/` trên Google Drive và upload vào đó:

- `archive.zip` — dataset Ano-AAD (9 GB)
- `abd_code.zip` — code, tạo trên máy bằng `scripts/pack_code.sh`

Chọn **Runtime → Change runtime type → T4 GPU** (hoặc tốt hơn), rồi **Run all**.

Kết quả ghi thẳng lên Drive (`MyDrive/abd/outputs`). Nếu Colab ngắt kết nối, chỉ cần **Run all** lại:
`track` và `extract` bỏ qua video đã xong. Cuối notebook tạo `MyDrive/abd/abd_results.zip` để tải về máy.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/abd'
!ls -lh {DRIVE}

## 1. Cài code

In [ ]:
!rm -rf /content/abd && mkdir -p /content/abd && unzip -q -o {DRIVE}/abd_code.zip -d /content/abd
%cd /content/abd
!pip install -q -e .
!abd --help | head -3

## 2. Chuẩn bị dataset (copy về đĩa local của Colab để đọc video nhanh)

In [ ]:
import os
if not os.path.isdir('/content/data/Ano-AAD/normal'):
    !cp {DRIVE}/archive.zip /content/archive.zip
    !bash scripts/prepare_ano_aad.sh /content/archive.zip /content/data/Ano-AAD
    !rm -f /content/archive.zip
else:
    print('dataset already prepared')

In [ ]:
CFG = '--config configs/colab.yaml'
!abd {CFG} index
!abd {CFG} split

## 3. YOLO + ByteTrack

Render 5 video có bbox vào `MyDrive/abd/outputs/vis/` — **mở xem vài cái** để chắc tracking ổn
(ID không nhảy liên tục, không bỏ sót người rõ ràng).

In [ ]:
!abd {CFG} track --render 5

## 4. Trích feature S3D (luồng frame + person)

In [ ]:
!abd {CFG} extract

## 5. (Tuỳ chọn) Train trên Colab

Train head trên feature rất nhẹ — có thể bỏ qua và train trên máy. 5-fold CV cho từng chế độ đầu vào
để so baseline full-frame với luồng person.

In [ ]:
for mode in ['frame', 'person', 'both']:
    !abd {CFG} --set model.input={mode} cv --name cv_{mode} --folds 5

In [ ]:
# model cuối cùng (train/val/test split cố định) — chọn mode tốt nhất từ bảng CV ở trên
BEST_MODE = 'both'
!abd {CFG} --set model.input={BEST_MODE} train --name {BEST_MODE}

## 6. Đóng gói kết quả để tải về máy

In [ ]:
%cd /content/drive/MyDrive/abd/outputs
!rm -f {DRIVE}/abd_results.zip
!zip -q -r {DRIVE}/abd_results.zip manifest.csv splits.json tracks features runs -x '*.mp4'
!ls -lh {DRIVE}/abd_results.zip
%cd /content/abd

Tải `MyDrive/abd/abd_results.zip` về thư mục project trên máy rồi chạy:

```bash
scripts/import_results.sh abd_results.zip
```